 results from both results and sprint table , only facts table , not dimensions

In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/gold-helper

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
df_results = (spark.table(f"{catalog_name}.{silver_schema}.results")
              .filter(F.col('batch_id') == v_batch_id)
              .withColumn('session_type' , F.lit('race'))
              .drop("race_name", "race_date", "ingestion_timestamp", "source_file", 'created_timestamp','updated_timestamp'))
df_sprints = (spark.table(f"{catalog_name}.{silver_schema}.sprints")
              .filter(F.col('batch_id') == v_batch_id)
              .withColumn('session_type' , F.lit('sprint'))
              .drop("race_name", "race_date", "ingestion_timestamp", "source_file", 'created_timestamp','updated_timestamp'))

In [0]:
results_sprints_df = df_results.unionByName(df_sprints)

In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("finish_position") == 1)
        .withColumn("is_podium", F.col("finish_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
write_to_gold(input_df=fact_session_results_df,
    target_table=target_table,
    merge_condition="""
        t.season = s.season
        AND t.round = s.round
        AND t.constructor_id = s.constructor_id
        AND t.driver_id = s.driver_id
        AND t.session_type = s.session_type
    """,
    columns_to_update=[
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "finish_position",
        "finish_position_text",
        "status",
        "is_win",
        "is_podium",
        "has_points"
    ]
)